## 1. Setup Awal: Import Library dan Definisi Konstanta

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ftfy sentence-transformers faiss-cpu pandas numpy scikit-learn beautifulsoup4

In [ ]:
## Setup Awal: Import Library dan Definisi Konstanta

# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import re
import os
import torch
import faiss
import unicodedata
import ftfy
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from IPython.display import display

print("Library berhasil di-import.")

# Definisi konstanta dan path file
DRIVE_PATH = '/content/drive/MyDrive/Skripsi Kode/'
DATASET_PATH = os.path.join(DRIVE_PATH, 'GoodReads_100k_books_2.csv')
OUTPUT_DIR = os.path.join(DRIVE_PATH, 'OutputArtefak/')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Path untuk menyimpan artefak yang telah diproses
# File ini KHUSUS untuk tampilan di aplikasi Flask
PROCESSED_METADATA_FOR_APP_PATH = os.path.join(OUTPUT_DIR, 'df_metadata_for_app.csv')

# File ini untuk pelatihan TF-IDF, menyertakan combined_text
DATA_FOR_TFIDF_TRAINING_PATH = os.path.join(OUTPUT_DIR, 'df_data_for_tfidf_training.csv')

NORMALIZED_EMBEDDINGS_PATH = os.path.join(OUTPUT_DIR, 'goodreads_embeddings_normalized.npy')
FAISS_INDEX_PATH = os.path.join(OUTPUT_DIR, 'goodreads_faiss_ivfflat.index')

# Konfigurasi model dan kolom data
SBERT_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
RELEVANT_TEXT_COLS = ['title', 'author', 'desc']
COLS_TO_KEEP_FOR_DISPLAY = ['title', 'author', 'img', 'link', 'rating', 'genre', 'desc']

print("Konstanta dan path telah didefinisikan.")

Library berhasil di-import.
Konstanta dan path telah didefinisikan.


## 2. Pemuatan dan Pra-pemrosesan Data

In [ ]:
# Memuat dataset dari file CSV
try:
    # Try different encodings
    try:
        df = pd.read_csv(DATASET_PATH, encoding='utf-8')
        print("Dataset berhasil dimuat dengan encoding UTF-8.")
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(DATASET_PATH, encoding='latin1')
            print("Dataset berhasil dimuat dengan encoding latin1.")
        except UnicodeDecodeError:
            df = pd.read_csv(DATASET_PATH, encoding='cp1252')
            print("Dataset berhasil dimuat dengan encoding cp1252.")

    print(f"Jumlah baris awal: {len(df)}")
    df.info()
except FileNotFoundError:
    print(f"ERROR: File dataset tidak ditemukan di {DATASET_PATH}")
    exit()
except Exception as e:
    print(f"ERROR: Terjadi kesalahan saat memuat dataset: {e}")
    exit()

/tmp/ipython-input-3016437681.py:9: DtypeWarning: Columns (8,9,10,12,13,14,15,16,17,18,20,23,27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATASET_PATH, encoding='latin1')


Dataset berhasil dimuat dengan encoding latin1.
Jumlah baris awal: 100021
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100021 entries, 0 to 100020
Data columns (total 29 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   author        100021 non-null  object 
 1   bookformat    96778 non-null   object 
 2   desc          93249 non-null   object 
 3   genre         89538 non-null   object 
 4   img           96974 non-null   object 
 5   isbn          85520 non-null   object 
 6   isbn13        88566 non-null   object 
 7   link          100018 non-null  object 
 8   pages         100005 non-null  object 
 9   rating        100005 non-null  object 
 10  reviews       100005 non-null  object 
 11  title         100019 non-null  object 
 12  totalratings  100005 non-null  object 
 13  Unnamed: 13   1 non-null       object 
 14  Unnamed: 14   1 non-null       object 
 15  Unnamed: 15   1 non-null       object 
 16  Unnamed: 16   1 no

In [ ]:
# --- Fungsi Pembersihan Teks  ---

def create_manual_replacement_dict():
    """
    Membuat dictionary untuk penggantian manual karakter yang sering salah encoding.
    Daftar ini dibuat berdasarkan analisis eksplorasi data.
    """
    # Pengurutan dari pola yang lebih panjang ke pendek penting untuk akurasi
    replacements = {
        # Pola Umum Mojibake (UTF-8 dibaca sebagai Windows-1252)
        'Ã¢â‚¬â„¢': "'",  # Apostrof kanan tunggal
        'Ã¢â‚¬â€': "—",  # Em dash
        'Ã¢â‚¬Å“': '"',  # Tanda kutip kiri ganda
        'Ã¢â‚¬Â': '"',   # Tanda kutip kanan ganda
        'Ã¢â‚¬Ëœ': "'",  # Tanda kutip kiri tunggal
        'Ã¢â‚¬Â¦': '...', # Elipsis
        'Ã‚Â': ' ',     # Non-breaking space

        # Karakter Eropa Tengah/Timur
        'Ã„â€¡': 'ć', 'Ã„â€': 'Ć', 'Ã„Â': 'č', 'Ã„Å’': 'Č', 'Ã„â€˜': 'đ',
        'Ã„Â': 'Đ', 'Ã…Â¡': 'š', 'Ã…Â': 'Š', 'Ã…Â¾': 'ž', 'Ã…Â½': 'Ž',

        # Karakter Skandinavia
        'ÃƒÂ¥': 'å', 'Ãƒâ€¦': 'Å', 'ÃƒÂ¤': 'ä', 'Ãƒâ€ž': 'Ä', 'ÃƒÂ¶': 'ö',
        'Ãƒâ€“': 'Ö', 'ÃƒÂ¸': 'ø', 'ÃƒËœ': 'Ø', 'ÃƒÂ¦': 'æ', 'Ãƒâ€': 'Æ',

        # Karakter Roman umum (Prancis, Jerman, Spanyol, dll.)
        'ÃƒÂ': 'à', 'Ãƒâ‚¬': 'À', 'ÃƒÂ¢': 'â', 'Ãƒâ€š': 'Â', 'ÃƒÂ§': 'ç',
        'Ãƒâ€¡': 'Ç', 'ÃƒÂ¨': 'è', 'ÃƒË†': 'È', 'ÃƒÂ©': 'é', 'Ãƒâ€°': 'É',
        'ÃƒÂª': 'ê', 'ÃƒÅ': 'Ê', 'ÃƒÂ«': 'ë', 'Ãƒâ€¹': 'Ë', 'ÃƒÂ®': 'î',
        'ÃƒÅ½': 'Î', 'ÃƒÂ¯': 'ï', 'Ãƒâ€': 'Ï', 'ÃƒÂ´': 'ô', 'Ãƒâ€': 'Ô',
        'ÃƒÂ¹': 'ù', 'Ãƒâ„¢': 'Ù', 'ÃƒÂ»': 'û', 'Ãƒâ€º': 'Û', 'ÃƒÂ¼': 'ü',
        'ÃƒÅ“': 'Ü', 'ÃƒÂ±': 'ñ', 'Ãƒâ€˜': 'Ñ', 'ÃƒÅ¸': 'ß',

        # Pola sisa
        'Â': ' ',
    }
    return replacements

# Inisialisasi daftar penggantian manual
MANUAL_REPLACEMENTS = create_manual_replacement_dict()

def clean_text(text_input):
    """
    Fungsi komprehensif untuk membersihkan teks mentah.
    """
    if not isinstance(text_input, str):
        return ""

    # 1. Perbaiki encoding umum dengan ftfy
    text_output = ftfy.fix_text(text_input)

    # 2. Hapus tag HTML
    text_output = BeautifulSoup(text_output, "html.parser").get_text(separator=" ")

    # 3. Normalisasi Unicode (NFC)
    text_output = unicodedata.normalize('NFC', text_output)

    # 4. Terapkan penggantian manual dari dictionary
    for bad_pattern, good_char in MANUAL_REPLACEMENTS.items():
        text_output = text_output.replace(bad_pattern, good_char)

    # 5. Konversi ke huruf kecil
    text_output = text_output.lower()

    # 6. Hapus karakter kontrol
    text_output = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text_output)

    # 7. Hapus spasi berlebih
    text_output = re.sub(r'\s+', ' ', text_output).strip()

    return text_output

In [ ]:
# Membuat salinan DataFrame untuk diproses
if 'df' in locals() and df is not None:
    df_processed = df.copy()
    df_processed['book_id'] = df_processed.index

    print("Memulai proses pembersihan dan persiapan teks...")

    # Definisikan kolom yang akan diproses
    RELEVANT_TEXT_COLS = ['title', 'author', 'desc']

    # PERBAIKI ENCODING PADA KOLOM ASLI
    print(" -> Memperbaiki encoding pada kolom tampilan (title, author, dll)...")
    for col in COLS_TO_KEEP_FOR_DISPLAY: # Gunakan kolom untuk tampilan
        if col in df_processed.columns: # Ensure the column exists
            df_processed[col] = df_processed[col].fillna('').apply(lambda x: ftfy.fix_text(str(x)))


    # BUAT TEKS BERSIH UNTUK MODEL
    print(" -> Membuat teks bersih gabungan untuk input model SBERT...")

    # Ambil teks yang sudah diperbaiki encodingnya, lalu terapkan pembersihan penuh.
    df_processed['combined_text'] = (
        df_processed['title'].fillna('') + ' ' +
        df_processed['author'].fillna('') + ' ' +
        df_processed['desc'].fillna('')
    ).apply(clean_text)


    # SIAPKAN METADATA FINAL UNTUK DISIMPAN
    # File CSV ini untuk TAMPILAN
    cols_to_save_for_display = ['book_id'] + COLS_TO_KEEP_FOR_DISPLAY
    # Ensure all columns in cols_to_save_for_display exist in df_processed
    existing_cols_to_save = [col for col in cols_to_save_for_display if col in df_processed.columns]
    df_final_metadata_for_display = df_processed[existing_cols_to_save]


    # SIAPKAN INPUT UNTUK SEL BERIKUTNYA
    # Siapkan list dari 'combined_text' yang sudah diproses penuh untuk SBERT
    corpus_texts = df_processed['combined_text'].tolist()

    print(f"Pra-pemrosesan selesai. Siap untuk generasi embedding. Jumlah teks: {len(corpus_texts)}")
else:
    print("ERROR: DataFrame 'df' tidak terdefinisi atau kosong. Tidak dapat melanjutkan pra-pemrosesan.")

Memulai proses pembersihan dan persiapan teks...
 -> Memperbaiki encoding pada kolom tampilan (title, author, dll)...
 -> Membuat teks bersih gabungan untuk input model SBERT...
Pra-pemrosesan selesai. Siap untuk generasi embedding. Jumlah teks: 100021


### Contoh Hasil Pra-pemrosesan Teks

In [ ]:
# Memuat data yang berisi combined_text
try:
    df_for_tfidf_training = pd.read_csv(DATA_FOR_TFIDF_TRAINING_PATH, encoding='utf-8')
    print(f"Data untuk pelatihan TF-IDF berhasil dimuat. Jumlah baris: {len(df_for_tfidf_training)}")

    # Menampilkan beberapa contoh hasil pra-pemrosesan teks
    # print("\nBeberapa contoh 'combined_text' setelah pra-pemrosesan:")
    # display(df_for_tfidf_training[['book_id', 'combined_text']].head())

    print("\nBeberapa contoh 'combined_text' acak setelah pra-pemrosesan:")
    display(df_for_tfidf_training[['book_id', 'combined_text']].sample(5))

except FileNotFoundError:
    print(f"ERROR: File data untuk pelatihan TF-IDF tidak ditemukan di {DATA_FOR_TFIDF_TRAINING_PATH}")
except Exception as e:
    print(f"ERROR: Terjadi kesalahan saat memuat atau menampilkan data: {e}")

Data untuk pelatihan TF-IDF berhasil dimuat. Jumlah baris: 100000

Beberapa contoh 'combined_text' acak setelah pra-pemrosesan:


,book_id,combined_text
80015,80015,te porta e shën pjetrit fatos kongoli dy histo...
12681,12681,"the race beat: the press, the civil rights str..."
66298,66298,old roses david austin covers all the most wor...
44102,44102,the twelfth heart elizabeth-irene baitie
50162,50162,structural package designs pepin press packagi...


## 3. Generasi Embedding dengan SBERT

In [ ]:
#GENERASI EMBEDDING
print(f"Memuat model SBERT: {SBERT_MODEL_NAME}...")
device = "cuda" if torch.cuda.is_available() else "cpu"
sbert_model = SentenceTransformer(SBERT_MODEL_NAME, device=device)
print(f"Model berhasil dimuat pada device: {sbert_model.device}")

print(f"Memulai generasi embedding untuk {len(corpus_texts)} teks...")
book_embeddings = sbert_model.encode(
    corpus_texts, batch_size=256, show_progress_bar=True, convert_to_numpy=True
)

book_embeddings_normalized = normalize(book_embeddings, norm='l2', axis=1).astype(np.float32)
print(f"Generasi dan normalisasi embedding selesai. Shape: {book_embeddings_normalized.shape}")

Memuat model SBERT: paraphrase-multilingual-MiniLM-L12-v2...
Model berhasil dimuat pada device: cuda:0
Memulai generasi embedding untuk 100021 teks...


Batches:   0%|          | 0/391 [00:00<?, ?it/s]

Generasi dan normalisasi embedding selesai. Shape: (100021, 384)


### Verifikasi Norma Vektor Setelah Normalisasi

In [ ]:
# Verifikasi Norma Vektor Setelah Normalisasi

print("\nMemverifikasi norma vektor setelah normalisasi...")

try:
    if 'book_embeddings_normalized' not in locals() or book_embeddings_normalized is None:
        book_embeddings_normalized = np.load(NORMALIZED_EMBEDDINGS_PATH)
        print(f"Embeddings ternormalisasi dimuat kembali dari: {NORMALIZED_EMBEDDINGS_PATH}")
    else:
        print("Menggunakan embeddings ternormalisasi yang sudah ada di memori.")

    # Pilih beberapa vektor acak untuk diperiksa normanya
    num_samples = 5
    if book_embeddings_normalized.shape[0] < num_samples:
        num_samples = book_embeddings_normalized.shape[0] # Ambil semua jika kurang dari 5

    sample_indices = np.random.choice(book_embeddings_normalized.shape[0], num_samples, replace=False)
    sample_embeddings = book_embeddings_normalized[sample_indices]

    # Hitung norma L2 untuk setiap sampel
    l2_norms = np.linalg.norm(sample_embeddings, axis=1)

    print(f"\nNorma L2 dari {num_samples} vektor embedding ternormalisasi terpilih:")
    for i, norm in zip(sample_indices, l2_norms):
        print(f" - Vektor indeks {i}: {norm:.6f}")

    # Periksa apakah semua norma mendekati 1
    tolerance = 1e-6 # Toleransi untuk floating point comparison
    all_norms_are_one = np.all(np.abs(l2_norms - 1.0) < tolerance)

    if all_norms_are_one:
        print("\nSemua norma vektor yang diperiksa mendekati 1.0 (sesuai dengan normalisasi L2).")
    else:
        print("\nBeberapa norma vektor yang diperiksa tidak mendekati 1.0. Perlu dicek proses normalisasi.")

except FileNotFoundError:
    print(f"ERROR: File embeddings ternormalisasi tidak ditemukan di {NORMALIZED_EMBEDDINGS_PATH}. Pastikan sel penyimpanan sudah dijalankan.")
except Exception as e:
    print(f"ERROR: Terjadi kesalahan saat memverifikasi norma: {e}")


Memverifikasi norma vektor setelah normalisasi...
Menggunakan embeddings ternormalisasi yang sudah ada di memori.

Norma L2 dari 5 vektor embedding ternormalisasi terpilih:
 - Vektor indeks 39000: 1.000000
 - Vektor indeks 56472: 1.000000
 - Vektor indeks 15865: 1.000000
 - Vektor indeks 86003: 1.000000
 - Vektor indeks 57249: 1.000000

Semua norma vektor yang diperiksa mendekati 1.0 (sesuai dengan normalisasi L2).


## 4. Pembangunan Indeks FAISS

In [ ]:
#PEMBANGUNAN FAISS
embedding_dimension = book_embeddings_normalized.shape[1]
n_total_vectors = book_embeddings_normalized.shape[0]
nlist = int(4 * np.sqrt(n_total_vectors))

print(f"Konfigurasi FAISS: Dimensi={embedding_dimension}, nlist={nlist}")
quantizer = faiss.IndexFlatIP(embedding_dimension)
index_ivf = faiss.IndexIVFFlat(quantizer, embedding_dimension, nlist, faiss.METRIC_INNER_PRODUCT)

print("Melatih indeks FAISS...")
index_ivf.train(book_embeddings_normalized)
print(f"Indeks FAISS sudah dilatih: {index_ivf.is_trained}")

print("Menambahkan embedding ke dalam indeks...")
index_ivf.add(book_embeddings_normalized)
print(f"Pembangunan indeks selesai. Total vektor: {index_ivf.ntotal}")

Konfigurasi FAISS: Dimensi=384, nlist=1265
Melatih indeks FAISS...
Indeks FAISS sudah dilatih: True
Menambahkan embedding ke dalam indeks...
Pembangunan indeks selesai. Total vektor: 100021


## 5. Penyimpanan Artefak

In [ ]:
print("\\nMenyimpan semua artefak yang telah dibuat...")

# 1. Simpan metadata UNTUK TAMPILAN APLIKASI (tanpa combined_text)
df_final_metadata_for_display.to_csv(PROCESSED_METADATA_FOR_APP_PATH, index=False, encoding='utf-8')
print(f" -> Metadata untuk aplikasi disimpan ke: {PROCESSED_METADATA_FOR_APP_PATH}")

# 2. Simpan data UNTUK PELATIHAN TF-IDF (dengan combined_text)
df_for_tfidf_training = df_processed[['book_id', 'combined_text']]
df_for_tfidf_training.to_csv(DATA_FOR_TFIDF_TRAINING_PATH, index=False, encoding='utf-8')
print(f" -> Data untuk pelatihan TF-IDF disimpan ke: {DATA_FOR_TFIDF_TRAINING_PATH}")

# 3. Simpan embedding yang sudah dinormalisasi
np.save(NORMALIZED_EMBEDDINGS_PATH, book_embeddings_normalized)
print(f" -> Embeddings ternormalisasi disimpan ke: {NORMALIZED_EMBEDDINGS_PATH}")

# 4. Simpan indeks FAISS
faiss.write_index(index_ivf, FAISS_INDEX_PATH)
print(f" -> Indeks FAISS disimpan ke: {FAISS_INDEX_PATH}")

print("Penyimpanan semua artefak selesai.")
print("--- Proses Persiapan Data dan Indeks Selesai ---")

\nMenyimpan semua artefak yang telah dibuat...
 -> Metadata untuk aplikasi disimpan ke: /content/drive/MyDrive/Skripsi Kode/OutputArtefak/df_metadata_for_app.csv
 -> Data untuk pelatihan TF-IDF disimpan ke: /content/drive/MyDrive/Skripsi Kode/OutputArtefak/df_data_for_tfidf_training.csv
 -> Embeddings ternormalisasi disimpan ke: /content/drive/MyDrive/Skripsi Kode/OutputArtefak/goodreads_embeddings_normalized.npy
 -> Indeks FAISS disimpan ke: /content/drive/MyDrive/Skripsi Kode/OutputArtefak/goodreads_faiss_ivfflat.index
Penyimpanan semua artefak selesai.
--- Proses Persiapan Data dan Indeks Selesai ---


## 6. Fungsi Pencarian dan Pengujian

In [ ]:
# Fungsi untuk melakukan pencarian semantik
def search_books(query_text, k=10):
    """
    Melakukan pencarian buku berdasarkan query teks.
    Kueri juga akan dibersihkan menggunakan fungsi `clean_text` yang sama.
    """
    if not query_text:
        return pd.DataFrame()

    # PENTING: Bersihkan kueri dengan fungsi yang sama seperti data korpus
    cleaned_query = clean_text(query_text)

    query_embedding = sbert_model.encode([cleaned_query], convert_to_numpy=True).astype(np.float32)
    query_embedding_normalized = normalize(query_embedding, norm='l2', axis=1)

    # Pastikan index_ivf global tersedia
    global index_ivf
    distances, indices = index_ivf.search(query_embedding_normalized, k)

    valid_indices = indices.flatten()[indices.flatten() != -1]
    if len(valid_indices) == 0:
        return pd.DataFrame()

    global df_display_data
    results_df = df_display_data.iloc[valid_indices].copy()
    results_df['similarity_score'] = distances.flatten()[indices.flatten() != -1]

    return results_df.sort_values(by='similarity_score', ascending=False)[['book_id', 'title', 'author', 'rating', 'genre', 'similarity_score']]

In [ ]:
# --- Pengujian ---
print("Memuat kembali artefak untuk pengujian...")
df_display_data = pd.read_csv(PROCESSED_METADATA_FOR_APP_PATH, encoding='utf-8')
index_ivf = faiss.read_index(FAISS_INDEX_PATH)

# Contoh 1
query1 = "a story about a young wizard going to a magic school"
print(f"\n-- Hasil Pencarian untuk: '{query1}' --")
display(search_books(query1, k=5))

# Contoh 2
query2 = "The Lord of the Rings by J.R.R. Tolkien"
print(f"\n-- Hasil Pencarian untuk: '{query2}' --")
display(search_books(query2, k=5))

# Contoh 3
query3 = "psychological thriller with a shocking twist"
print(f"\n-- Hasil Pencarian untuk: '{query3}' --")
display(search_books(query3, k=5))

Memuat kembali artefak untuk pengujian...


/tmp/ipython-input-3374882089.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_display_data = pd.read_csv(PROCESSED_METADATA_FOR_APP_PATH, encoding='utf-8')



-- Hasil Pencarian untuk: 'a story about a young wizard going to a magic school' --


,book_id,title,author,rating,genre,similarity_score
52642,52642,The Scholomance,R. Lee Smith,3.87,"Fantasy,Horror,Dark,Paranormal,Demons,Fantasy,...",0.750886
47332,47332,Kaytek the Wizard,"Janusz Korczak,Antonia Lloyd-Jones",3.73,"Fantasy,Childrens,Fiction,European Literature,...",0.739208
89285,89285,Crossroads and the Himalayan Crystals,C. Toni Graham,4.32,"Contemporary,Drama,Epic",0.733718
60654,60654,Magician: Apprentice,Raymond E. Feist,4.14,"Fantasy,Fiction,Fantasy,Epic Fantasy,Fantasy,H...",0.714694
69265,69265,"Magical X Miracle, Vol. 1","Yuzu Mizutani,Yoohae Yang,Mark Ilvedson",4.03,"Sequential Art,Manga,Fantasy,Sequential Art,Gr...",0.689849



-- Hasil Pencarian untuk: 'The Lord of the Rings by J.R.R. Tolkien' --


,book_id,title,author,rating,genre,similarity_score
73164,73164,The Ring Sets Out,J.R.R. Tolkien,4.3,"Fantasy,Classics,Fiction,Science Fiction Fanta...",0.796885
33361,33361,The Ring Goes South,J.R.R. Tolkien,4.39,"Fantasy,Classics,Fiction,Audiobook,Science Fic...",0.782561
33362,33362,The Ring Goes East,J.R.R. Tolkien,4.4,"Fantasy,Fiction,Classics,Science Fiction Fanta...",0.780081
71967,71967,The Lord of the Rings: Official Movie Guide,Brian Sibley,4.46,"Nonfiction,Media Tie In,Reference,Culture,Film...",0.759833
71860,71860,The Lord of the Rings: The Fellowship of the R...,"Jude Fisher,J.R.R. Tolkien",4.62,"Fantasy,Fiction,Art,Culture,Film,Reference,Med...",0.747813



-- Hasil Pencarian untuk: 'psychological thriller with a shocking twist' --


,book_id,title,author,rating,genre,similarity_score
14587,14587,Shock Value: How a Few Eccentric Outsiders Gav...,Jason Zinoman,3.85,"Nonfiction,Horror,Culture,Film,History,Culture...",0.673140
17806,17806,A já pořád kdo to tluče,Radka Denemarková,3.98,"European Literature,Czech Literature",0.656498
24081,24081,Passion's Bright Fury,Radclyffe,4.22,"LGBT,Lesbian,LGBT,Romance,Romance,Lesbian Roma...",0.637214
11839,11839,Geheime kamers,Jeroen Brouwers,3.7,"Fiction,European Literature,Dutch Literature,R...",0.611739
63074,63074,Wonderland Avenue: Tales of Glamour and Excess,"Danny Sugerman,Ralph Steadman",4.23,"Music,Biography,Nonfiction,Autobiography,Memoi...",0.600291
